In [5]:
import pandas as pd
import re
from typing import Dict, List, Optional, Set
from collections import defaultdict
import gc
import os

class UnifiedFeatureMapper:
    """Optimized version for large datasets"""

    def __init__(self, data_dict_path: str, max_instances: Optional[int] = None):
        self.data_dict_path = data_dict_path
        self.max_instances = max_instances
        self.dict_df = None
        self.clean_mapping = {}
        self.relevant_feature_names = set()

    HEALTH_FOLDERS = [
        'Health-related outcomes > First occurrences',
        'Health-related outcomes > Cancer register',
        'Health-related outcomes > Death register',
        'Health-related outcomes > Algorithmically-defined outcomes',
    ]

    FMRI_FOLDERS = [
        'Brain imaging > Resting functional brain MRI',
        'Brain imaging > Task functional brain MRI'
    ]

    FMRI_DEMOGRAPHIC_FIELDS = {
        'p31',          # Sex
        'p21003_i2',    # Age at imaging visit (Instance 2)
        'p21003_i3',    # Age at imaging visit (Instance 3)
    }

    def load_and_process(self, data_file_path: Optional[str] = None) -> Dict[str, str]:
        if not self._load_data_dictionary():
            return {}

        self.relevant_feature_names = self._filter_relevant_features()
        print(f"✅ Identified {len(self.relevant_feature_names)} features from folders/demographics.")

        if self.max_instances is not None:
            self.relevant_feature_names = self._limit_instances(self.relevant_feature_names)
            print(f"✅ Limited to max {self.max_instances} instances for non-demographic fields. "
                  f"New count: {len(self.relevant_feature_names)}")

        if data_file_path:
            print("🔍 Intersecting with main data file columns...")
            data_columns = pd.read_csv(data_file_path, nrows=0).columns.tolist()
            self.relevant_feature_names &= set(data_columns)
            print(f"✅ Final feature count: {len(self.relevant_feature_names)}")

        raw_mapping = self._create_raw_mapping(list(self.relevant_feature_names))
        self.clean_mapping = self._extract_raw_titles(raw_mapping)
        self._print_summary(list(self.relevant_feature_names))
        return self.clean_mapping

    def _load_data_dictionary(self) -> bool:
        try:
            self.dict_df = pd.read_csv(
                self.data_dict_path,
                usecols=['name', 'title', 'folder_path'],
                low_memory=False
            )
            print(f"✓ Loaded data dictionary: {self.dict_df.shape}")
            return True
        except Exception as e:
            print(f"❌ Failed to load dictionary: {e}")
            return False

    def _filter_relevant_features(self) -> Set[str]:
        df = self.dict_df
        health_mask = df['folder_path'].str.startswith(tuple(self.HEALTH_FOLDERS), na=False)
        fMRI_mask = df['folder_path'].str.startswith(tuple(self.FMRI_FOLDERS), na=False)
        demo_mask = df['name'].isin(self.FMRI_DEMOGRAPHIC_FIELDS)
        final_mask = health_mask | fMRI_mask | demo_mask
        return set(df.loc[final_mask, 'name'].dropna().astype(str))

    def _limit_instances(self, feature_names: Set[str]) -> Set[str]:
        if self.max_instances is None or self.max_instances < 0:
            return feature_names

        protected_features = self.FMRI_DEMOGRAPHIC_FIELDS & feature_names
        limitable_features = feature_names - self.FMRI_DEMOGRAPHIC_FIELDS

        limited_features = set(protected_features)

        base_to_features = defaultdict(list)
        for name in limitable_features:
            match = re.search(r'^(.+?)(?:_i\d+|_a\d+|\(i\d+\)|\(a\d+\))$', name)
            base = match.group(1) if match else name
            base_to_features[base].append(name)

        for base, full_names in base_to_features.items():
            sorted_names = sorted(full_names)
            limited_features.update(sorted_names[:self.max_instances])

        return limited_features

    def _create_raw_mapping(self, feature_list: List[str]) -> Dict[str, str]:
        relevant = self.dict_df[self.dict_df['name'].isin(feature_list)]
        mapping_series = relevant.set_index('name')['title']
        return mapping_series.to_dict()

    def _extract_raw_titles(self, raw_mapping: Dict[str, str]) -> Dict[str, str]:
        final_mapping = {}
        for code, title in raw_mapping.items():
            if pd.isna(title):
                final_mapping[code] = code
            else:
                final_mapping[code] = str(title).strip()
        return final_mapping

    def _print_summary(self, feature_list: List[str]) -> None:
        mapped = len(self.clean_mapping)
        print(f"\n📊 Final Mapping: {mapped}/{len(feature_list)} features named")
        demo_vars = [f for f in feature_list if f in self.FMRI_DEMOGRAPHIC_FIELDS]
        print(f"   Demographics (protected): {len(demo_vars)} variables")
        for f in list(self.clean_mapping.keys())[:3]:
            print(f"     {f} → {self.clean_mapping[f]}")

    def get_relevant_columns(self) -> List[str]:
        return ['eid'] + sorted(self.relevant_feature_names)


# ICD helper functions (unchanged)
def extract_icd10_code_flexible(title: str) -> str:
    match1 = re.search(r'(?:Date\s+)?([A-Z]\d{2}(?:\.\d{1,3}[A-Z]?)?)\s+first\s+reported', title)
    if match1:
        return match1.group(1)[:3]
    paren_content_match = re.search(r'\(([^)]+)\)$', title)
    if paren_content_match:
        paren_content = paren_content_match.group(1)
        icd_match = re.search(r'([A-Z]\d{2}(?:\.\d{1,3}[A-Z]?)?)', paren_content)
        if icd_match:
            return icd_match.group(1)[:3]
    colon_match = re.search(r':\s*([A-Z]\d{2}(?:\.\d{1,3}[A-Z]?)?)(?:\s|$|,|\()', title)
    if colon_match:
        return colon_match.group(1)[:3]
    return None

def build_expanded_icd_map(icd_category_map: dict) -> dict:
    expanded_map = {}
    for key, category_name in icd_category_map.items():
        if '-' in key:
            parts = key.split('-')
            if len(parts) == 2:
                start, end = parts[0], parts[1]
                if start[0] == end[0]:
                    letter = start[0]
                    start_num = int(start[1:])
                    end_num = int(end[1:])
                    for num in range(start_num, end_num + 1):
                        expanded_map[f"{letter}{num:02d}"] = category_name
        else:
            expanded_map[key] = category_name
    return expanded_map

def process_chunk_with_categorization(chunk_df, clean_mapping, expanded_icd_map, 
                                     specific_disorders_map, demographic_cols):
    # Rename columns
    rename_dict = {col: clean_mapping[col] for col in chunk_df.columns if col in clean_mapping}
    chunk_renamed = chunk_df.rename(columns=rename_dict)
    
    # Initialize result with demographics
    result_cols = {}
    for col in demographic_cols:
        if col in chunk_renamed.columns:
            result_cols[col] = chunk_renamed[col]
    
    # Track columns by category
    columns_by_category = defaultdict(list)
    diagnosis_keywords = ['first reported', 'cause of death', 'algorithmically-defined', 
                         'cancer', 'myocardial infarction', 'stroke', 'diabetes']
    
    for col in chunk_renamed.columns:
        if any(kw in col.lower() for kw in diagnosis_keywords):
            icd_code = extract_icd10_code_flexible(col)
            if icd_code:
                category = expanded_icd_map.get(icd_code)
                if category:
                    columns_by_category[category].append(col)
    
    # Create broad category columns
    for category, cols in columns_by_category.items():
        if cols:
            result_cols[category] = chunk_renamed[cols].notna().any(axis=1).astype(int)
    
    # Create specific disorder columns
    for disorder_name, target_code in specific_disorders_map.items():
        matching_cols = []
        for col in chunk_renamed.columns:
            if any(kw in col.lower() for kw in diagnosis_keywords):
                icd_code = extract_icd10_code_flexible(col)
                if icd_code == target_code:
                    matching_cols.append(col)
        if matching_cols:
            result_cols[disorder_name] = chunk_renamed[matching_cols].notna().any(axis=1).astype(int)
    
    # >>> REMOVED: Do NOT preserve p40013_i* (death register) <<<
    # Cancer detection is handled correctly later using p20001/p40006
    
    return pd.DataFrame(result_cols)

def add_health_status(df, age_mri_path):
    """Merge age, detect cancer properly using p20001 (self-report) and p40006 (registry), drop rows without fMRI age."""
    # Load and merge age
    df_age = pd.read_csv(age_mri_path, usecols=['eid', 'p21003_i2'])
    df['eid'] = df['eid'].astype(str)
    df_age['eid'] = df_age['eid'].astype(str)
    df = pd.merge(df, df_age, on='eid', how='left')
    df = df.rename(columns={'p21003_i2': 'Age'})
    
    # Drop rows without Age
    initial_rows = len(df)
    df = df.dropna(subset=['Age']).copy()
    dropped = initial_rows - len(df)
    if dropped > 0:
        print(f"⚠️ Dropped {dropped} rows with missing Age (p21003_i2).")

    # >>> CORRECT CANCER DETECTION: p20001 (self-report) + p40006 (registry) <<<
    cancer_indicators = []

    # 1. Self-reported cancer (p20001_i*): any non-null = cancer
    p20001_cols = [col for col in df.columns if col.startswith('p20001_i')]
    if p20001_cols:
        has_cancer_self = df[p20001_cols].notna().any(axis=1)
        cancer_indicators.append(has_cancer_self)
        print(f"✅ Found {len(p20001_cols)} self-reported cancer columns (p20001_i*).")

    # 2. Cancer registry diagnosis (p40006_i*): any non-null = cancer
    p40006_cols = [col for col in df.columns if col.startswith('p40006_i')]
    if p40006_cols:
        has_cancer_registry = df[p40006_cols].notna().any(axis=1)
        cancer_indicators.append(has_cancer_registry)
        print(f"✅ Found {len(p40006_cols)} cancer registry columns (p40006_i*).")

    # Combine
    if cancer_indicators:
        df['has_cancer'] = pd.concat(cancer_indicators, axis=1).any(axis=1).astype(int)
        n_cancer = df['has_cancer'].sum()
        print(f"✅ Total participants with cancer (p20001 or p40006): {n_cancer}")
    else:
        df['has_cancer'] = 0
        print("⚠️ No cancer fields found. Setting has_cancer=0.")

    # Health status logic (unchanged)
    NEURO_PSYCH_COLS = [
        'Psychopathology_Dementia', 'Psychopathology_Organic_Mental_Disorder',
        'Psychopathology_Substance_Use', 'Psychopathology_Schizophrenia_Spectrum',
        'Psychopathology_Mood_Affective', 'Psychopathology_Other_Psych',
        'NervousSystem_Inflammatory_Infectious', 'NervousSystem_Degenerative_Huntington_Ataxia',
        'NervousSystem_Parkinsons_Other_Movement', 'NervousSystem_Dementia_Developmental',
        'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
        'NervousSystem_Epilepsy_Status_Epilepticus', 'NervousSystem_Cerebrovascular',
        'NervousSystem_Sleep_Disorders', 'NervousSystem_Other_Neuro',
        'Chronic_Sense_Organs',
        'ICD_F32_Depressive_Episode', 'ICD_F33_Recurrent_Depressive',
        'ICD_G20_Parkinsons', 'ICD_G21_Secondary_Parkinsonism',
        'ICD_G40_Epilepsy', 'ICD_G41_Status_Epilepticus',
        'ICD_F20_Schizophrenia', 'ICD_F31_Bipolar',
        'ICD_G10_Huntingtons', 'ICD_G11_HereditaryAtaxia',
        'ICD_G12_SpinalMuscularAtrophy', 'ICD_G24_Dystonia',
        'ICD_G23_OtherDegenerativeBasalGanglia', 'ICD_G25_OtherExtrapyramidal'
    ]
    neuro_cols = [c for c in NEURO_PSYCH_COLS if c in df.columns]

    SYSTEMIC_COLS = [
        'Chronic_Blood_Immune',
        'Endocrine_Nutritional_Metabolic_Diabetes',
        'Endocrine_Nutritional_Metabolic_Hypothyroidism',
        'Endocrine_Nutritional_Metabolic_Hyperthyroidism',
        'Endocrine_Nutritional_Metabolic_Other_Endocrine',
        'Chronic_Cardiovascular',
        'Chronic_Respiratory',
        'Chronic_Digestive',
        'Chronic_Genitourinary'
    ]
    systemic_cols = [c for c in SYSTEMIC_COLS if c in df.columns]

    df['neuro_healthy'] = (df[neuro_cols].fillna(0).max(axis=1) == 0).astype(int)
    strict_cols = neuro_cols + systemic_cols
    df['strict_healthy'] = ((df[strict_cols].fillna(0).max(axis=1) == 0) & (df['has_cancer'] == 0)).astype(int)

    # Enforce column order
    key_cols = ['eid', 'Sex', 'Age', 'neuro_healthy', 'strict_healthy', 'has_cancer']
    key_cols = [c for c in key_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in key_cols]
    df = df[key_cols + other_cols]

    return df


# ICD mappings (unchanged)
ICD_CATEGORY_MAP = {
    'F00': 'Psychopathology_Dementia', 'F01': 'Psychopathology_Dementia',
    'F02': 'Psychopathology_Dementia', 'F03': 'Psychopathology_Dementia',
    'F04': 'Psychopathology_Organic_Mental_Disorder', 'F05': 'Psychopathology_Organic_Mental_Disorder',
    'F06': 'Psychopathology_Organic_Mental_Disorder', 'F07': 'Psychopathology_Organic_Mental_Disorder',
    'F09': 'Psychopathology_Organic_Mental_Disorder',
    'F10': 'Psychopathology_Substance_Use', 'F11': 'Psychopathology_Substance_Use',
    'F12': 'Psychopathology_Substance_Use', 'F13': 'Psychopathology_Substance_Use',
    'F14': 'Psychopathology_Substance_Use', 'F15': 'Psychopathology_Substance_Use',
    'F16': 'Psychopathology_Substance_Use', 'F17': 'Psychopathology_Substance_Use',
    'F18': 'Psychopathology_Substance_Use', 'F19': 'Psychopathology_Substance_Use',
    'F20': 'Psychopathology_Schizophrenia_Spectrum', 'F21': 'Psychopathology_Schizophrenia_Spectrum',
    'F22': 'Psychopathology_Schizophrenia_Spectrum', 'F23': 'Psychopathology_Schizophrenia_Spectrum',
    'F24': 'Psychopathology_Schizophrenia_Spectrum', 'F25': 'Psychopathology_Schizophrenia_Spectrum',
    'F28': 'Psychopathology_Schizophrenia_Spectrum', 'F29': 'Psychopathology_Schizophrenia_Spectrum',
    'F30': 'Psychopathology_Mood_Affective', 'F31': 'Psychopathology_Mood_Affective',
    'F32': 'Psychopathology_Mood_Affective', 'F33': 'Psychopathology_Mood_Affective',
    'F34': 'Psychopathology_Mood_Affective', 'F99': 'Psychopathology_Other_Psych',
    'G00': 'NervousSystem_Inflammatory_Infectious', 'G04': 'NervousSystem_Inflammatory_Infectious',
    'G06': 'NervousSystem_Inflammatory_Infectious', 'G07': 'NervousSystem_Inflammatory_Infectious',
    'G08': 'NervousSystem_Inflammatory_Infectious', 'G09': 'NervousSystem_Inflammatory_Infectious',
    'G10': 'NervousSystem_Degenerative_Huntington_Ataxia', 'G11': 'NervousSystem_Degenerative_Huntington_Ataxia',
    'G12': 'NervousSystem_Degenerative_Huntington_Ataxia', 'G14': 'NervousSystem_Degenerative_Huntington_Ataxia',
    'G20': 'NervousSystem_Parkinsons_Other_Movement', 'G21': 'NervousSystem_Parkinsons_Other_Movement',
    'G22': 'NervousSystem_Parkinsons_Other_Movement', 'G23': 'NervousSystem_Parkinsons_Other_Movement',
    'G24': 'NervousSystem_Parkinsons_Other_Movement', 'G25': 'NervousSystem_Parkinsons_Other_Movement',
    'G30': 'NervousSystem_Dementia_Developmental', 'G31': 'NervousSystem_Dementia_Developmental',
    'G32': 'NervousSystem_Dementia_Developmental',
    'G35': 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
    'G36': 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
    'G37': 'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
    'G40': 'NervousSystem_Epilepsy_Status_Epilepticus', 'G41': 'NervousSystem_Epilepsy_Status_Epilepticus',
    'G45': 'NervousSystem_Cerebrovascular', 'G46': 'NervousSystem_Cerebrovascular',
    'G47': 'NervousSystem_Sleep_Disorders',
    'G50-G99': 'NervousSystem_Other_Neuro',
    'E03': 'Endocrine_Nutritional_Metabolic_Hypothyroidism',
    'E05': 'Endocrine_Nutritional_Metabolic_Hyperthyroidism',
    'E10': 'Endocrine_Nutritional_Metabolic_Diabetes', 'E11': 'Endocrine_Nutritional_Metabolic_Diabetes',
    'E16': 'Endocrine_Nutritional_Metabolic_Other_Pancreatic',
    'E20': 'Endocrine_Nutritional_Metabolic_Other_Endocrine',
    'E27': 'Endocrine_Nutritional_Metabolic_Other_Endocrine',
    'E40': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E41': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E43': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E44': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E45': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E46': 'Endocrine_Nutritional_Metabolic_Protein_Energy_Malnutrition',
    'E50-E60': 'Endocrine_Nutritional_Metabolic_Vitamin_Deficiency',
    'C00-C97': 'Chronic_Cancer', 'I00-I99': 'Chronic_Cardiovascular',
    'J00-J99': 'Chronic_Respiratory', 'K00-K95': 'Chronic_Digestive',
    'N00-N99': 'Chronic_Genitourinary', 'M00-M99': 'Chronic_Musculoskeletal',
    'L00-L99': 'Chronic_Skin_Subcutaneous', 'Q00-Q99': 'Chronic_Congenital',
    'R00-R99': 'Chronic_Symptoms_Signs', 'D50-D89': 'Chronic_Blood_Immune',
    'A00-B99': 'Infectious_Disease', 'H90': 'Chronic_Sense_Organs',
    'V01-Y98': 'External_Cause_Injury_Poisoning', 'P07': 'Chronic_Perinatal',
    'Q05': 'Chronic_Chromosomal',
}

SPECIFIC_DISORDERS = {
    'ICD_F32_Depressive_Episode': 'F32',
    'ICD_F33_Recurrent_Depressive': 'F33',
    'ICD_G20_Parkinsons': 'G20',
    'ICD_G21_Secondary_Parkinsonism': 'G21',
    'ICD_G40_Epilepsy': 'G40',
    'ICD_G41_Status_Epilepticus': 'G41',
    'ICD_F20_Schizophrenia': 'F20',
    'ICD_F31_Bipolar': 'F31',
    'ICD_G10_Huntingtons': 'G10',
    'ICD_G11_HereditaryAtaxia': 'G11',
    'ICD_G12_SpinalMuscularAtrophy': 'G12',
    'ICD_G24_Dystonia': 'G24',
    'ICD_G23_OtherDegenerativeBasalGanglia': 'G23',
    'ICD_G25_OtherExtrapyramidal': 'G25',
}


def process_large_csv_with_health_status(
    data_file_path, 
    data_dict_path, 
    age_mri_path,
    output_path, 
    icd_category_map, 
    specific_disorders_map, 
    chunk_size=10000, 
    max_instances=2
):
    print("🚀 Step 1: Building feature mapping...")
    mapper = UnifiedFeatureMapper(data_dict_path, max_instances=max_instances)
    mapping = mapper.load_and_process(data_file_path=data_file_path)
    columns_to_load = mapper.get_relevant_columns()
    
    print("\n🔄 Step 2: Building ICD category map...")
    expanded_icd_map = build_expanded_icd_map(icd_category_map)
    
    demographic_code_keys = [k for k in mapping.keys() if k in mapper.FMRI_DEMOGRAPHIC_FIELDS]
    demographic_cols = ['eid'] + [mapping[k] for k in demographic_code_keys]
    
    print(f"\n📊 Step 3: Processing file in chunks of {chunk_size} rows...")
    print(f"   Loading {len(columns_to_load)} columns")
    
    first_chunk = True
    total_rows = 0
    
    for chunk_num, chunk in enumerate(pd.read_csv(
        data_file_path,
        usecols=columns_to_load,
        chunksize=chunk_size,
        low_memory=False
    ), 1):
        print(f"   Processing chunk {chunk_num} ({len(chunk)} rows)...", end='\r')
        
        processed_chunk = process_chunk_with_categorization(
            chunk, 
            mapping,
            expanded_icd_map,
            specific_disorders_map,
            demographic_cols
        )
        
        if first_chunk:
            processed_chunk.to_csv(output_path, index=False, mode='w')
            first_chunk = False
        else:
            processed_chunk.to_csv(output_path, index=False, mode='a', header=False)
        
        total_rows += len(chunk)
        del chunk, processed_chunk
        gc.collect()
    
    print(f"\n✅ Raw processing complete! Total rows: {total_rows}")
    
    # >>> NEW: Load full output, add health status, and re-save <<<
    print("\n🚀 Step 4: Adding health status flags and merging with age_at_mri...")
    df_full = pd.read_csv(output_path)
    df_final = add_health_status(df_full, age_mri_path)
    df_final.to_csv(output_path, index=False)
    
    print(f"\n✅ Final dataset with health status saved to: {output_path}")
    print(f"   Shape: {df_final.shape}")
    print(f"   Neuro-healthy: {df_final['neuro_healthy'].sum()} ({df_final['neuro_healthy'].mean():.1%})")
    print(f"   Strictly healthy (incl. no cancer): {df_final['strict_healthy'].sum()} ({df_final['strict_healthy'].mean():.1%})")




In [6]:
# Run everything
if __name__ == "__main__":
    path = '/home/jaizor/jaizor/xtra/notebooks/UKBB/data/csv/_/'
    process_large_csv_with_health_status(
        data_file_path=path + 'data_field_four_cat.csv',
        data_dict_path=path + 'database.dataset.data_dictionary.csv',
        age_mri_path=path + 'age_at_mri.csv',
        output_path=path + 'fMRI_final_with_health_status.csv',
        icd_category_map=ICD_CATEGORY_MAP,
        specific_disorders_map=SPECIFIC_DISORDERS,
        chunk_size=10000,
        max_instances=2
    )

🚀 Step 1: Building feature mapping...
✓ Loaded data dictionary: (36015, 3)
✅ Identified 2515 features from folders/demographics.
✅ Limited to max 2 instances for non-demographic fields. New count: 2336
🔍 Intersecting with main data file columns...
✅ Final feature count: 2334

📊 Final Mapping: 2334/2334 features named
   Demographics (protected): 1 variables
     p31 → Sex
     p40000_i0 → Date of death | Instance 0
     p40000_i1 → Date of death | Instance 1

🔄 Step 2: Building ICD category map...

📊 Step 3: Processing file in chunks of 10000 rows...
   Loading 2335 columns
   Processing chunk 51 (1981 rows)....
✅ Raw processing complete! Total rows: 501981

🚀 Step 4: Adding health status flags and merging with age_at_mri...
⚠️ Dropped 411352 rows with missing Age (p21003_i2).
⚠️ No cancer fields found. Setting has_cancer=0.

✅ Final dataset with health status saved to: /home/jaizor/jaizor/xtra/notebooks/UKBB/data/csv/_/fMRI_final_with_health_status.csv
   Shape: (90629, 53)
   Neuro-h